# CNN-BiLSTM-MHA + PPO Trader — Training (V4)

Full training pipeline for the CNN-BiLSTM-MHA (forecaster) and PPO (agent) models,
with improved Reward V2, vectorization for 2 GPUs, and TensorFlow.js export.

**Changes from V3 → V4:**
- **Soft labels** (sigmoid of future return) replace hard 0/1 binary targets
- **Wavelet decomposition** (db4, level-2) adds 6 frequency-domain features
  - Trend deviation, medium-freq oscillation, high-freq noise level
  - Trend slope, noise/signal energy ratio, spectral entropy
- NUM_FEATURES: 10 → 16

**Unchanged:** CNN-BiLSTM-MHA architecture, PPO agent, reward V2, vectorized runner.


In [ ]:
# Training deps + tensorflowjs (Kaggle image tolerates installing it at the top).
# On Kaggle the pre-built image plays well with tensorflowjs's TF pin, so no need
# to defer it to the conversion cell (that workaround is only required on Colab).
!pip install -q ccxt tensorflowjs PyWavelets


In [ ]:
import math, time, os, json
from contextlib import nullcontext
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import ccxt

# Hyperparameters
WINDOW_SIZE          = 128            # 128 x 15min ~ 32h of context
HORIZON              = 4              # predict direction 4 x 15min = 60 min ahead
HIDDEN_UNITS         = 128
DROPOUT              = 0.2
NUM_FEATURES         = 16             # 10 original + 6 wavelet
L2                   = 1e-5
LR                   = 5e-4
MIN_LR               = 1e-5
EARLY_STOP_PATIENCE  = 15
BATCH_SIZE           = 256
FORECAST_EPOCHS      = 100

# PPO
STATE_SIZE   = 13
ACTION_SIZE  = 3       # 0=HOLD 1=BUY 2=SELL
GAMMA        = 0.99
CLIP_RATIO   = 0.2
POLICY_LR    = 3e-4
VALUE_LR     = 1e-3
PPO_MIN_LR   = 1e-5
LR_DECAY     = 0.99
MAX_GRAD_NORM = 0.5
PPO_EPOCHS   = 10

# Constants
INDICATOR_WARMUP      = 200
RETURN_CLIP           = 0.1
VOLATILITY_WINDOW     = 20
FEE_RATE              = 0.00075
BACKTEST_HOLDOUT      = 1000
VALIDATION_FRACTION   = 0.1
EMBARGO_FRACTION      = 0.01
HISTORICAL_CANDLES    = 35_000        # 35k x 15min ~ 365 days

# Risk - matches .env values used at inference. Stop/take based on 15m ATR.
STOP_LOSS_PCT    = 0.0030             # 0.30%
TAKE_PROFIT_PCT  = 0.0050             # 0.50%, R:R 1.67
SLIPPAGE_PCT     = 0.0005

# SOFT-LABEL CLASSIFICATION MODE:
# Labels are no longer hard 0/1. Instead, each label is a soft probability
# proportional to the magnitude of the future return, via sigmoid transform.
# A +0.001% move → label ≈ 0.52 (nearly uncertain)
# A +2.0%  move → label ≈ 0.98 (very confident bullish)
# A -1.5%  move → label ≈ 0.05 (very confident bearish)
# This gives the model richer gradient signal on strong moves
# while naturally down-weighting ambiguous noise near zero.
SOFT_LABEL_SCALE     = 400            # sigmoid(excess_return * SCALE): graded labels, avoid saturation/collapse
SOFT_LABEL_CLIP      = (0.02, 0.98)   # avoid extremes for numeric stability
#
# Targets are soft floats in [0.02, 0.98] (sigmoid of future return).
# pred_ret in reward function is P(up) in [0,1].
# pred_strength = (pred_ret - 0.5) is the signed directional confidence.
PRED_STRENGTH_OPPORTUNITY_TH = 0.05   # P(up) > 0.55 = strong enough to regret HOLD
LOSS_CUT_THRESHOLD = STOP_LOSS_PCT / 3.0  # unchanged: cut loss bonus on realized PnL

FORECASTER_SYMBOLS = [
    "BTC/USDT", "ETH/USDT",
    "SOL/USDT", "LINK/USDT", "BNB/USDT",
    "XRP/USDT", "DOGE/USDT"
]
AGENT_SYMBOL       = "BTC/USDT"
TIMEFRAME          = "15m"

# Multi-GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detected: {len(gpus)} -> {[g.name for g in gpus]}")

if len(gpus) > 1:
    strategy = tf.distribute.MirroredStrategy()
else:
    strategy = tf.distribute.get_strategy()

NUM_REPLICAS = strategy.num_replicas_in_sync
GLOBAL_BATCH = BATCH_SIZE * NUM_REPLICAS
print(f"Replicas: {NUM_REPLICAS}, global batch: {GLOBAL_BATCH}")

# On a single device the default-strategy scope can corrupt the CUDA stream on
# some Colab TF builds (CUDA_ERROR_INVALID_HANDLE during model build). It adds
# nothing with one GPU, so use a no-op context unless we truly have >1 replica.
def dist_scope():
    return strategy.scope() if strategy.num_replicas_in_sync > 1 else nullcontext()
import os

# Portable working dir: Kaggle, Colab, or local. All outputs go under WORK_DIR.
if os.path.isdir("/kaggle/working"):
    WORK_DIR = "/kaggle/working"
elif os.path.isdir("/content/drive/MyDrive"):
    WORK_DIR = "/content/drive/MyDrive/AI Trading"
    os.makedirs(WORK_DIR, exist_ok=True)
else:
    WORK_DIR = "."
print(f"WORK_DIR = {WORK_DIR}")


In [ ]:
# ── GPU SMOKE TEST ───────────────────────────────────────────────────────────
# Minimal check that the CUDA context is healthy BEFORE building the model.
# If the cast below raises CUDA_ERROR_INVALID_HANDLE, the problem is the runtime
# environment (TF/CUDA mismatch or a corrupted context), NOT the model code:
#   1) Runtime > Disconnect and delete runtime (a plain Restart is not enough),
#   2) reconnect with a GPU, run this cell FIRST (before any extra pip install).
#   3) if it passes here but fails after the install cell, the install is the cause.
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
print("Strategy replicas:", strategy.num_replicas_in_sync if "strategy" in globals() else "(run the setup cell above first)")
with tf.device('/GPU:0'):
    _probe = tf.cast(tf.constant([1.0, 2.0]), tf.float32)
    print("GPU cast OK:", _probe.numpy())

In [ ]:
# Load dataset built by the separate 'ai-spot-trading - Dataset' notebook.
# Kaggle: add this dataset via '+ Add Data' -> appears at /kaggle/input/<slug>/dataset.npz
# Colab:  uses kagglehub (requires ~/.kaggle/kaggle.json uploaded once).

DATASET_SLUG = 'acaurangel/ai-spot-trading-dataset'
_dataset_name = DATASET_SLUG.split('/')[-1]
_kaggle_input_path = f'/kaggle/input/{_dataset_name}/dataset.npz'

if os.path.isfile(_kaggle_input_path):
    dataset_path = _kaggle_input_path
else:
    try:
        import kagglehub
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
        import kagglehub
    _dataset_dir = kagglehub.dataset_download(DATASET_SLUG)
    dataset_path = os.path.join(_dataset_dir, 'dataset.npz')

print(f'Loading dataset from {dataset_path}')
_data = np.load(dataset_path)
X_train           = _data['X_train']
y_train           = _data['y_train']
X_val             = _data['X_val']
y_val             = _data['y_val']
agent_X           = _data['agent_X']
agent_candles     = _data['agent_candles']
agent_vol_samples = _data['agent_vol_samples']
print(f'  X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'  X_val:   {X_val.shape},  y_val:   {y_val.shape}')
print(f'  agent_X: {agent_X.shape}, agent_candles: {agent_candles.shape}, agent_vol: {agent_vol_samples.shape}')

In [ ]:
# ── Multi-Head Self-Attention with Residual + LayerNorm (Keras 3) ────────────
# Replaces the old single-head SelfAttentionLayer.
#
# Why this is better:
#   - Multiple heads (4) capture different temporal patterns simultaneously
#     (trend, mean-reversion, breakout, consolidation)
#   - Residual connection prevents gradient degradation in deeper stacks
#   - LayerNorm stabilizes training and improves convergence speed
#   - Uses native Keras layer → no custom_objects needed for TF.js export
#
# We keep it wrapped in a single layer so build_bilstm stays clean.

class MultiHeadAttentionBlock(layers.Layer):
    """Multi-Head Attention + Add & Norm (Transformer encoder block, no FFN)."""

    def __init__(self, d_model: int, num_heads: int = 4,
                 dropout: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model   = d_model
        self.num_heads = num_heads
        self.drop_rate = dropout

    def build(self, input_shape):
        feat_dim = input_shape[-1]

        # Project input to d_model if dimensions don't match (for residual)
        self.needs_projection = (feat_dim != self.d_model)
        if self.needs_projection:
            self.input_proj = layers.Dense(self.d_model, name="mha_input_proj")

        self.mha = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.d_model // self.num_heads,
            dropout=self.drop_rate,
            name="mha_core",
        )
        self.layernorm = layers.LayerNormalization(name="mha_layernorm")
        self.dropout   = layers.Dropout(self.drop_rate)
        super().build(input_shape)

    def call(self, x, training=None):
        residual = self.input_proj(x) if self.needs_projection else x
        attn_out = self.mha(query=x, value=x, key=x, training=training)
        attn_out = self.dropout(attn_out, training=training)
        return self.layernorm(residual + attn_out)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "d_model":   self.d_model,
            "num_heads":  self.num_heads,
            "dropout":    self.drop_rate,
        })
        return cfg

print("MultiHeadAttentionBlock OK (replaces SelfAttentionLayer).")


In [ ]:
# ── BiLSTM Baseline (CLASSIFICATION) ──────────────────────────────────────────
#
# Architecture changes:
#   - Removed Conv1D multi-scale (CNN) and MaxPooling.
#   - Removed MultiHeadAttentionBlock (MHA).
#   - Established a stable BiLSTM-only baseline to reduce dimensionality.
#
# The input/output shapes are unchanged:
#   Input:  (batch, WINDOW_SIZE, NUM_FEATURES)
#   Output: (batch, HORIZON)                    [sigmoid]
#
# So the PPO agent, feature builder, and TF.js inference remain untouched.

def build_bilstm(seq_len=128, num_features=16, hidden=64,
                 dropout=0.2, horizon=4, l2_val=1e-5):
    reg = regularizers.L2(l2_val)
    inp = keras.Input(shape=(seq_len, num_features))

    # ── Stage 1: BiLSTM temporal modeling (Directly from input) ─────────
    x = layers.Bidirectional(
        layers.LSTM(hidden, return_sequences=True,
                    kernel_regularizer=reg, recurrent_regularizer=reg),
        name="bilstm_1",
    )(inp)
    x = layers.Dropout(dropout, name="bilstm1_dropout")(x)

    x = layers.Bidirectional(
        layers.LSTM(hidden // 2, return_sequences=True,
                    kernel_regularizer=reg, recurrent_regularizer=reg),
        name="bilstm_2",
    )(x)
    x = layers.Dropout(dropout, name="bilstm2_dropout")(x)

    # ── Stage 2: Classification head ───────────────────────────────────
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    x = layers.Dense(hidden, kernel_regularizer=reg,
                     bias_initializer=keras.initializers.Constant(0.01),
                     name="fc_hidden")(x)
    x = layers.LeakyReLU(negative_slope=0.01, name="fc_act")(x)
    x = layers.Dropout(dropout, name="fc_dropout")(x)

    # Sigmoid: independent P(up) per horizon step
    out = layers.Dense(horizon, activation="sigmoid", name="classifier_head")(x)

    return keras.Model(inputs=inp, outputs=out,
                       name="Baseline_BiLSTM_Classifier")

with dist_scope():
    forecaster = build_bilstm(
        WINDOW_SIZE, NUM_FEATURES, HIDDEN_UNITS, DROPOUT, HORIZON, L2
    )

print(f"\nModel: {forecaster.name}")
print(f"  Architecture: Pure BiLSTM Baseline (No CNN, No MHA)")
print(f"  Total params: {forecaster.count_params():,}")
forecaster.summary()

In [ ]:
# CNN-BiLSTM-MHA training — binary cross-entropy on SOFT labels + early stopping
# BCE works natively with soft targets in [0,1] — no code change needed.
# The model sees richer gradient signal: strong moves get steep gradients,
# ambiguous moves near 0.5 get shallow gradients (natural down-weighting).

class CosineAnnealingCallback(keras.callbacks.Callback):
    def __init__(self, lr_max, lr_min, total_epochs):
        super().__init__()
        self.lr_max = lr_max
        self.lr_min = lr_min
        self.total  = total_epochs

    def on_epoch_begin(self, epoch, logs=None):
        cosine = 0.5 * (1 + math.cos(math.pi * epoch / self.total))
        lr = self.lr_min + (self.lr_max - self.lr_min) * cosine
        self.model.optimizer.learning_rate.assign(lr)


# ── Anti-collapse loss: BCE + variance penalty ───────────────────────────────
# With soft labels centered near 0.5, the BCE-optimal trivial solution is to
# predict the constant mean (predStd -> 0). This penalty term kicks in when the
# batch-wise std of predictions falls below COLLAPSE_MIN_STD, pushing the model
# off the degenerate constant attractor so it has to actually use the inputs.
COLLAPSE_MIN_STD = 0.05      # require batch predStd >= 0.05 (>>0.002 we are seeing)
COLLAPSE_WEIGHT  = 20.0      # penalty magnitude when fully collapsed (std=0)

@keras.utils.register_keras_serializable(package="custom")
def collapse_aware_bce(y_true, y_pred):
    """Standard BCE plus a hinge penalty on low batch std of predictions."""
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))
    # Std across the batch dimension, per horizon, then average across horizons.
    std_per_horizon = tf.math.reduce_std(y_pred, axis=0)
    avg_std = tf.reduce_mean(std_per_horizon)
    collapse_penalty = tf.maximum(0.0, COLLAPSE_MIN_STD - avg_std) * COLLAPSE_WEIGHT
    return bce + collapse_penalty

# ── Metrics that actually work with SOFT labels ──────────────────────────────
# Keras' built-in "accuracy"/AUC expect binary y_true; our labels are soft floats
# in [0.02, 0.98], so they degenerate (AUC printed 0.0 every epoch). These binarize
# y_true at 0.5 on horizon 0 (the next-bar signal, == SIGNAL_HORIZON_INDEX on TS side).
SIGNAL_INDEX = 0

class DirectionalAccuracy(keras.metrics.Metric):
    """% of bars where predicted direction (P(up)>0.5) matches the true soft label's direction."""
    def __init__(self, index=SIGNAL_INDEX, name="dir_acc", **kw):
        super().__init__(name=name, **kw)
        self.index = index
        self.total = self.add_weight(name="total", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")
    def update_state(self, y_true, y_pred, sample_weight=None):
        yt = tf.cast(y_true[:, self.index] > 0.5, tf.float32)
        yp = tf.cast(y_pred[:, self.index] > 0.5, tf.float32)
        match = tf.cast(tf.equal(yt, yp), tf.float32)
        self.total.assign_add(tf.reduce_sum(match))
        self.count.assign_add(tf.cast(tf.size(match), tf.float32))
    def result(self):
        return self.total / (self.count + 1e-8)
    def reset_state(self):
        self.total.assign(0.0)
        self.count.assign(0.0)

class BinaryAUCFromSoft(keras.metrics.AUC):
    """Standard AUC but binarizes the soft label (y_true>0.5) on horizon 0 before scoring."""
    def __init__(self, index=SIGNAL_INDEX, name="auc", **kw):
        super().__init__(name=name, **kw)
        self.index = index
    def update_state(self, y_true, y_pred, sample_weight=None):
        yt = tf.cast(y_true[:, self.index] > 0.5, tf.int32)
        yp = y_pred[:, self.index]
        return super().update_state(yt, yp, sample_weight)

with dist_scope():
    forecaster.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss=collapse_aware_bce,
        metrics=[DirectionalAccuracy(name="dir_acc"), BinaryAUCFromSoft(name="auc")]
    )

callbacks = [
    CosineAnnealingCallback(LR, MIN_LR, FORECAST_EPOCHS),
    keras.callbacks.EarlyStopping(
        monitor="val_dir_acc", patience=EARLY_STOP_PATIENCE,
        min_delta=0.001, mode="max", restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        f"{WORK_DIR}/bilstm_best.keras",
        monitor="val_dir_acc", mode="max", save_best_only=True, verbose=1
    ),
]

print(f"Training BiLSTM (CLASSIFICATION) -- {FORECAST_EPOCHS} max epochs (batch={GLOBAL_BATCH})...")
print("Watch val_dir_acc (>0.50 = beats coin flip) and val_auc (>0.50 = real signal). "
      "Ignore val_loss magnitude: with soft labels its floor is ~0.69 regardless of skill.")
history = forecaster.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=FORECAST_EPOCHS,
    batch_size=GLOBAL_BATCH,
    shuffle=True,
    callbacks=callbacks,
    verbose=1,
)
print("BiLSTM trained.")


In [ ]:
# ── Forecaster sanity check (mirrors src/application/services/ForecastSanityCheck.ts) ──
# After BiLSTM training, validate that predictions actually vary (predStd > 0) and
# that they have a real directional edge (directionalAccuracy > 50%). With soft
# labels the loss alone is uninformative (floor ~0.69), so this gate is what tells
# us whether the forecaster is usable BEFORE we burn a session retraining the PPO.

SANITY_SAMPLES = 200

# Use the most recent SANITY_SAMPLES feature windows + their next-bar ground truth.
_n   = len(agent_X)
_end = _n - 1                                  # leave one bar for the future-close lookup
_start = max(0, _end - SANITY_SAMPLES)
sample_idx = np.arange(_start, _end)

# Forecaster eager call (avoids MirroredStrategy splitting the batch).
_preds  = forecaster(agent_X[sample_idx], training=False).numpy()  # (B, HORIZON)
_preds0 = _preds[:, 0]                                              # horizon 0 == SIGNAL_INDEX

# Actual next-bar direction (close[t+1] > close[t]).
_anchor_close = agent_candles[sample_idx + WINDOW_SIZE, 4]
_next_close   = agent_candles[sample_idx + WINDOW_SIZE + 1, 4]
_actual_up    = (_next_close > _anchor_close).astype(np.int32)

pred_mean       = float(np.mean(_preds0))
pred_std        = float(np.std(_preds0))
actual_up_rate  = float(np.mean(_actual_up))
_predicted_up   = (_preds0 > 0.5).astype(np.int32)
directional_acc = float(np.mean(_predicted_up == _actual_up))

print("Forecaster sanity check (classification):")
print(f"  samples:              {len(sample_idx)}")
print(f"  predMean:             {pred_mean:.3f}")
print(f"  predStd:              {pred_std:.3f}")
print(f"  actualUpRate:         {actual_up_rate:.3f}")
print(f"  directionalAccuracy:  {directional_acc * 100:.2f}%")

DEGENERATE_STD = 1e-4
TARGET_STD     = 0.02
TARGET_DIR_ACC = 0.51
if pred_std < DEGENERATE_STD:
    print("  [BLOCK] predStd < 1e-4 — forecaster collapsed. Do NOT retrain the PPO.")
elif pred_std < TARGET_STD:
    print(f"  [WARN] predStd < {TARGET_STD} — signal magnitude below the reward noise floor; the PPO may not extract this edge.")
elif directional_acc < TARGET_DIR_ACC:
    print(f"  [WARN] directionalAccuracy < {TARGET_DIR_ACC*100:.0f}% — review before PPO.")
else:
    print("  [OK] Forecaster has dispersion AND a real edge — safe to proceed to PPO training.")

In [ ]:
# ── Fase 2: CNN-BiLSTM-MHA ──────────────────────────────────────────
CNN_FILTERS   = 32
CNN_KERNELS   = [3, 5, 7]
CNN_POOL_SIZE = 2

def build_cnn_bilstm_mha(seq_len=128, num_features=16, hidden=64,
                         dropout=0.2, horizon=4, l2_val=1e-5):
    reg = regularizers.L2(l2_val)
    inp = keras.Input(shape=(seq_len, num_features))

    # ── Stage 1: Conv1D multi-scale feature extraction ──────────────────
    conv_branches = []
    for k in CNN_KERNELS:
        branch = layers.Conv1D(
            filters=CNN_FILTERS, kernel_size=k,
            padding="same", activation="relu",
            kernel_regularizer=reg,
            name=f"conv1d_k{k}",
        )(inp)
        branch = layers.BatchNormalization(name=f"bn_k{k}")(branch)
        conv_branches.append(branch)

    x = layers.Concatenate(name="cnn_concat")(conv_branches)
    x = layers.MaxPooling1D(pool_size=CNN_POOL_SIZE, name="cnn_pool")(x)
    x = layers.Dropout(dropout, name="cnn_dropout")(x)

    # ── Stage 2: BiLSTM temporal modeling ───────────────────────────────
    x = layers.Bidirectional(
        layers.LSTM(hidden, return_sequences=True,
                    kernel_regularizer=reg, recurrent_regularizer=reg),
        name="bilstm_1",
    )(x)
    x = layers.Dropout(dropout, name="bilstm1_dropout")(x)

    x = layers.Bidirectional(
        layers.LSTM(hidden // 2, return_sequences=True,
                    kernel_regularizer=reg, recurrent_regularizer=reg),
        name="bilstm_2",
    )(x)
    x = layers.Dropout(dropout, name="bilstm2_dropout")(x)

    # ── Stage 3: Multi-Head Attention + Residual + LayerNorm ────────────
    x = MultiHeadAttentionBlock(
        d_model=hidden, num_heads=4, dropout=dropout,
        name="mha_block",
    )(x)

    # ── Stage 4: Classification head ───────────────────────────────────
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    x = layers.Dense(hidden, kernel_regularizer=reg,
                     bias_initializer=keras.initializers.Constant(0.01),
                     name="fc_hidden")(x)
    x = layers.LeakyReLU(negative_slope=0.01, name="fc_act")(x)
    x = layers.Dropout(dropout, name="fc_dropout")(x)

    out = layers.Dense(horizon, activation="sigmoid", name="classifier_head")(x)

    return keras.Model(inputs=inp, outputs=out, name="CNN_BiLSTM_MHA_Full")

with dist_scope():
    full_forecaster = build_cnn_bilstm_mha(
        WINDOW_SIZE, NUM_FEATURES, HIDDEN_UNITS, DROPOUT, HORIZON, L2
    )
    full_forecaster.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss=collapse_aware_bce,
        metrics=[DirectionalAccuracy(name="dir_acc"), BinaryAUCFromSoft(name="auc")]
    )

full_callbacks = [
    CosineAnnealingCallback(LR, MIN_LR, FORECAST_EPOCHS),
    keras.callbacks.EarlyStopping(
        monitor="val_dir_acc", patience=EARLY_STOP_PATIENCE,
        min_delta=0.001, mode="max", restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        f"{WORK_DIR}/full_model_best.keras",
        monitor="val_dir_acc", mode="max", save_best_only=True, verbose=1
    ),
]

print(f"\nTreinando CNN-BiLSTM-MHA Completo -- {FORECAST_EPOCHS} epochs max...")
full_history = full_forecaster.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=FORECAST_EPOCHS,
    batch_size=GLOBAL_BATCH,
    shuffle=True,
    callbacks=full_callbacks,
    verbose=1,
)

# ── Pre-compute agent forecasts in batch ─────────────────────────────────────
CHUNK = 512
agent_forecasts = []
n_samples = len(agent_X)
print(f"\nGenerating {n_samples} forecasts in batch using FULL model...")
for start in range(0, n_samples, CHUNK):
    batch = agent_X[start : start + CHUNK]
    preds = full_forecaster(batch, training=False).numpy()
    agent_forecasts.append(preds)
agent_forecasts = np.concatenate(agent_forecasts, axis=0)
print(f"Pre-computed forecasts: {agent_forecasts.shape}")


In [ ]:
# ── Sanity check for full_forecaster (the model that ACTUALLY feeds the PPO) ──
# Same diagnostic as the baseline check above, but validates the CNN-BiLSTM-MHA
# Full model whose predictions become agent_forecasts -> input to the PPO state.
# This is the number that matters before retraining the PPO.

SANITY_SAMPLES = 200
_n   = len(agent_X)
_end = _n - 1
_start = max(0, _end - SANITY_SAMPLES)
sample_idx = np.arange(_start, _end)

_preds  = full_forecaster(agent_X[sample_idx], training=False).numpy()
_preds0 = _preds[:, 0]

_anchor_close = agent_candles[sample_idx + WINDOW_SIZE, 4]
_next_close   = agent_candles[sample_idx + WINDOW_SIZE + 1, 4]
_actual_up    = (_next_close > _anchor_close).astype(np.int32)

pred_mean       = float(np.mean(_preds0))
pred_std        = float(np.std(_preds0))
actual_up_rate  = float(np.mean(_actual_up))
_predicted_up   = (_preds0 > 0.5).astype(np.int32)
directional_acc = float(np.mean(_predicted_up == _actual_up))

print("Full forecaster sanity check (the one feeding the PPO):")
print(f"  samples:              {len(sample_idx)}")
print(f"  predMean:             {pred_mean:.3f}")
print(f"  predStd:              {pred_std:.3f}")
print(f"  actualUpRate:         {actual_up_rate:.3f}")
print(f"  directionalAccuracy:  {directional_acc * 100:.2f}%")

DEGENERATE_STD = 1e-4
TARGET_STD     = 0.02
TARGET_DIR_ACC = 0.51
if pred_std < DEGENERATE_STD:
    print("  [BLOCK] predStd < 1e-4 — full_forecaster collapsed. Do NOT retrain the PPO.")
elif pred_std < TARGET_STD:
    print(f"  [WARN] predStd < {TARGET_STD} — signal magnitude below the reward noise floor; PPO may not extract this edge.")
elif directional_acc < TARGET_DIR_ACC:
    print(f"  [WARN] directionalAccuracy < {TARGET_DIR_ACC*100:.0f}% — review before PPO.")
else:
    print("  [OK] full_forecaster has dispersion AND a real edge — safe to proceed to PPO training.")

## PPO — Reward V2 + Multi-GPU + Vectorization

Changes from the original pipeline:
- **Reward V2**: no idle penalty; profit bonus; forecaster-based opportunity cost; progressive hold-loser penalty
- **Entropy bonus** (0.01) on actor loss to prevent policy collapse
- **Vectorization**: 16 parallel episodes per update (~12x speedup over sequential)
- **Best checkpoint**: automatically saves the best model during training


In [ ]:
# Reward V2 + helper functions (CLASSIFICATION inputs)

# Reward V2 hyperparameters
PROFIT_BONUS       = 0.5    # extra multiplier for profitable sells
OPPORTUNITY_COST   = 0.3    # weight of opportunity cost when flat
HOLD_WINNER_BONUS  = 1.5    # multiplier when holding a winning position
HOLD_LOSER_DECAY   = 0.25   # growing penalty for holding a losing position
LOSS_CUT_BONUS     = 0.001  # bonus for cutting a loss before stop-loss

def assemble_state(candles, i, forecast, position, entry_price, bars_in_pos, volatility):
    close = candles[i, 4]
    open_ = candles[i, 1]
    high  = candles[i, 2]
    low   = candles[i, 3]
    pnl   = (close - entry_price) / entry_price if position == 1 and entry_price > 0 else 0.0
    return [
        1 - position, pnl, min(bars_in_pos / 100.0, 1.0),
        (open_ - close) / close, (high - close) / close,
        (low - close) / close, (high - low) / close,
        position, volatility, *forecast[:4],
    ]

def apply_exit_guards(action, position, entry_price, current_close, sl, tp):
    if position != 1:
        return action
    pnl = (current_close - entry_price) / entry_price if entry_price > 0 else 0.0
    if pnl <= -sl or pnl >= tp:
        return 2
    return action

def compute_reward_v2(candles, i, action, position, entry_price, bars_in_pos, forecast):
    """Reward V2 - classification mode.

    forecast[0] is now P(up) in [0,1]. pred_strength is the signed
    directional confidence centered at 0 (positive = bullish view).
    """
    close      = candles[i, 4]
    next_close = candles[i + 1, 4]
    price_ret  = (next_close - close) / close
    pred_ret   = forecast[0]                  # probability in [0, 1]
    pred_strength = pred_ret - 0.5            # signed confidence in [-0.5, +0.5]

    # FLAT (no position)
    if position == 0:
        if action == 1:  # BUY
            base = price_ret - FEE_RATE - SLIPPAGE_PCT
            if pred_strength > 0:
                base += pred_strength * OPPORTUNITY_COST
            return base
        else:  # HOLD flat
            if pred_strength > PRED_STRENGTH_OPPORTUNITY_TH:
                return -pred_strength * OPPORTUNITY_COST * 0.3
            return 0.0

    # IN POSITION
    pnl = (close - entry_price) / entry_price if entry_price > 0 else 0.0

    if action == 2:  # SELL
        realized = pnl
        base = realized - 2 * FEE_RATE - 2 * SLIPPAGE_PCT
        if realized > 0:
            base += realized * PROFIT_BONUS
        elif realized < -LOSS_CUT_THRESHOLD and realized > -STOP_LOSS_PCT:
            base += LOSS_CUT_BONUS
        return base

    # HOLD in position
    if pnl > 0:
        return price_ret * HOLD_WINNER_BONUS
    else:
        time_factor = min(bars_in_pos / 20.0, 1.0)
        hold_penalty = -abs(pnl) * HOLD_LOSER_DECAY * time_factor
        return price_ret + hold_penalty

print("Reward V2 + helper functions OK (classification inputs).")
print(f"  Profit bonus: {PROFIT_BONUS}x | Opportunity cost: {OPPORTUNITY_COST}")
print(f"  Pred-strength opportunity threshold: {PRED_STRENGTH_OPPORTUNITY_TH:.3f} (P(up) > {0.5 + PRED_STRENGTH_OPPORTUNITY_TH:.2f})")

In [ ]:
# ── Actor / Critic (inside strategy.scope, optimizers pre-built) ─────────────

def build_actor(state_size=13, action_size=3):
    inp = keras.Input(shape=(state_size,))
    x   = layers.Dense(128, activation="tanh")(inp)
    x   = layers.Dense(64,  activation="tanh")(x)
    out = layers.Dense(action_size, activation="softmax")(x)
    return keras.Model(inp, out, name="Actor")

def build_critic(state_size=13):
    inp = keras.Input(shape=(state_size,)
)
    x   = layers.Dense(128, activation="tanh")(inp)
    x   = layers.Dense(64,  activation="tanh")(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name="Critic")

with dist_scope():
    actor  = build_actor(STATE_SIZE, ACTION_SIZE)
    critic = build_critic(STATE_SIZE)
    policy_opt = keras.optimizers.Adam(POLICY_LR)
    value_opt  = keras.optimizers.Adam(VALUE_LR)
    policy_opt.build(actor.trainable_variables)
    value_opt.build(critic.trainable_variables)

update_count = 0

def get_current_lr(base_lr, decay, count):
    return max(base_lr * (decay ** count), PPO_MIN_LR)

def sample_action(probs):
    return np.random.choice(len(probs), p=probs)

def decide(state_vec):
    s = tf.convert_to_tensor([state_vec], dtype=tf.float32)
    probs = actor(s, training=False).numpy()[0]
    value = critic(s, training=False).numpy()[0, 0]
    action = sample_action(probs)
    log_prob = math.log(probs[action] + 1e-8)
    return action, log_prob, value

print("Actor/Critic OK (built inside strategy scope).")
actor.summary()

In [ ]:
# ── PPO update functions with Batched Distributed Strategy ──
PPO_BATCH_SIZE = 8192
ENTROPY_COEFF = 0.01

def clip_grads(grads, max_norm):
    return [tf.clip_by_norm(g, max_norm) for g in grads]

@tf.function
def distributed_update_actor(states, actions, advantages, old_log_probs):
    def update_actor_step(st, ac, adv, olp):
        with tf.GradientTape() as tape:
            probs     = actor(st, training=True)
            log_probs = tf.math.log(probs + 1e-8)
            indices   = tf.stack([tf.range(tf.shape(ac)[0]), ac], axis=1)
            new_lp    = tf.gather_nd(log_probs, indices)
            ratio     = tf.exp(new_lp - olp)
            surr1     = ratio * adv
            surr2     = tf.clip_by_value(ratio, 1 - CLIP_RATIO, 1 + CLIP_RATIO) * adv
            policy_loss = -tf.reduce_mean(tf.minimum(surr1, surr2))
            entropy = -tf.reduce_mean(tf.reduce_sum(probs * log_probs, axis=1))
            loss = policy_loss - ENTROPY_COEFF * entropy
        grads = tape.gradient(loss, actor.trainable_variables)
        grads = clip_grads(grads, MAX_GRAD_NORM)
        policy_opt.apply_gradients(zip(grads, actor.trainable_variables))
        return loss, entropy

    per_replica_loss, per_replica_entropy = strategy.run(update_actor_step, args=(states, actions, advantages, old_log_probs))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_loss, axis=None), strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_entropy, axis=None)

@tf.function
def distributed_update_critic(states, returns):
    def update_critic_step(st, ret):
        with tf.GradientTape() as tape:
            values = tf.squeeze(critic(st, training=True), axis=1)
            loss   = tf.reduce_mean(tf.square(ret - values))
        grads = tape.gradient(loss, critic.trainable_variables)
        grads = clip_grads(grads, MAX_GRAD_NORM)
        value_opt.apply_gradients(zip(grads, critic.trainable_variables))
        return loss

    per_replica_loss = strategy.run(update_critic_step, args=(states, returns))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_loss, axis=None)

def compute_advantages_vectorized(rewards, values, terminals, gamma=0.99, gae_lambda=0.95):
    n_steps = rewards.shape[0]
    advantages = np.zeros_like(rewards, dtype=np.float32)
    last_adv = np.zeros(rewards.shape[1], dtype=np.float32)
    for t in reversed(range(n_steps)):
        if t == n_steps - 1:
            next_val = np.zeros(rewards.shape[1], dtype=np.float32)
        else:
            next_val = values[t+1]
        mask = 1.0 - terminals[t]
        delta = rewards[t] + gamma * next_val * mask - values[t]
        last_adv = delta + gamma * gae_lambda * mask * last_adv
        advantages[t] = last_adv
    return advantages

def compute_returns_vectorized(rewards, terminals, gamma=0.99):
    n_steps = rewards.shape[0]
    returns = np.zeros_like(rewards, dtype=np.float32)
    last_ret = np.zeros(rewards.shape[1], dtype=np.float32)
    for t in reversed(range(n_steps)):
        mask = 1.0 - terminals[t]
        last_ret = rewards[t] + gamma * mask * last_ret
        returns[t] = last_ret
    return returns

def ppo_update_batched(states_np, actions_np, log_probs_np, values_np, rewards_np, terminals_np):
    global update_count

    adv_np = compute_advantages_vectorized(rewards_np, values_np, terminals_np, GAMMA)
    ret_np = compute_returns_vectorized(rewards_np, terminals_np, GAMMA)

    flat_states = states_np.reshape(-1, states_np.shape[-1])
    flat_actions = actions_np.reshape(-1)
    flat_old_lp = log_probs_np.reshape(-1)
    flat_adv = adv_np.reshape(-1)
    flat_ret = ret_np.reshape(-1)

    dataset_size = flat_states.shape[0]
    indices = np.arange(dataset_size)

    p_loss_sum, p_ent_sum, v_loss_sum = 0.0, 0.0, 0.0
    steps = 0

    for _ in range(PPO_EPOCHS):
        np.random.shuffle(indices)
        for start in range(0, dataset_size, PPO_BATCH_SIZE):
            end = start + PPO_BATCH_SIZE
            idx_batch = indices[start:end]

            b_states = tf.constant(flat_states[idx_batch], dtype=tf.float32)
            b_actions = tf.constant(flat_actions[idx_batch], dtype=tf.int32)
            b_old_lp = tf.constant(flat_old_lp[idx_batch], dtype=tf.float32)
            b_adv = tf.constant(flat_adv[idx_batch], dtype=tf.float32)
            b_ret = tf.constant(flat_ret[idx_batch], dtype=tf.float32)

            l, e = distributed_update_actor(b_states, b_actions, b_adv, b_old_lp)
            vl = distributed_update_critic(b_states, b_ret)

            p_loss_sum += l.numpy()
            p_ent_sum += e.numpy()
            v_loss_sum += vl.numpy()
            steps += 1

    update_count += 1
    new_p_lr = get_current_lr(POLICY_LR, LR_DECAY, update_count)
    new_v_lr = get_current_lr(VALUE_LR,  LR_DECAY, update_count)
    policy_opt.learning_rate.assign(new_p_lr)
    value_opt.learning_rate.assign(new_v_lr)

    return p_loss_sum/steps, p_ent_sum/steps, v_loss_sum/steps

print(f"Distributed Batched PPO update OK. (batch_size={PPO_BATCH_SIZE}, entropy_coeff={ENTROPY_COEFF})")


In [ ]:
# Vectorized episode runner V2 - CLASSIFICATION inputs

NUM_PARALLEL = 512

@tf.function
def tf_run_episodes_vectorized(closes, opens, highs, lows, forecasts, vol_samples, n_envs, n_steps, current_fee_rate):
    states_ta    = tf.TensorArray(tf.float32, size=n_steps)
    actions_ta   = tf.TensorArray(tf.int32,   size=n_steps)
    log_probs_ta = tf.TensorArray(tf.float32, size=n_steps)
    values_ta    = tf.TensorArray(tf.float32, size=n_steps)
    rewards_ta   = tf.TensorArray(tf.float32, size=n_steps)

    positions    = tf.zeros([n_envs], dtype=tf.int32)
    entry_prices = tf.zeros([n_envs], dtype=tf.float32)
    bars         = tf.zeros([n_envs], dtype=tf.int32)

    for idx in tf.range(n_steps):
        i = idx + WINDOW_SIZE

        close = closes[i]
        open_ = opens[i]
        high  = highs[i]
        low   = lows[i]
        vol   = vol_samples[idx]
        forecast = forecasts[idx]

        close_batch = tf.fill([n_envs], close)
        open_batch  = tf.fill([n_envs], open_)
        high_batch  = tf.fill([n_envs], high)
        low_batch   = tf.fill([n_envs], low)
        vol_batch   = tf.fill([n_envs], vol)
        forecast_batch = tf.tile(tf.expand_dims(forecast[:4], 0), [n_envs, 1])

        safe_entry = tf.where(entry_prices > 0.0, entry_prices, 1.0)
        pnls = tf.where((positions == 1) & (entry_prices > 0.0), (close_batch - safe_entry) / safe_entry, 0.0)

        st_1 = tf.cast(1 - positions, tf.float32)
        st_2 = pnls
        st_3 = tf.minimum(tf.cast(bars, tf.float32) / 100.0, 1.0)
        st_4 = (open_batch - close_batch) / close_batch
        st_5 = (high_batch - close_batch) / close_batch
        st_6 = (low_batch - close_batch) / close_batch
        st_7 = (high_batch - low_batch) / close_batch
        st_8 = tf.cast(positions, tf.float32)
        st_9 = vol_batch

        states = tf.concat([
            tf.expand_dims(st_1, 1), tf.expand_dims(st_2, 1), tf.expand_dims(st_3, 1),
            tf.expand_dims(st_4, 1), tf.expand_dims(st_5, 1), tf.expand_dims(st_6, 1),
            tf.expand_dims(st_7, 1), tf.expand_dims(st_8, 1), tf.expand_dims(st_9, 1),
            forecast_batch
        ], axis=1)

        probs  = actor(states, training=False)
        values = tf.squeeze(critic(states, training=False), axis=1)

        u = tf.random.uniform(tf.shape(probs), minval=0.00001, maxval=0.99999)
        actions = tf.argmax(tf.math.log(probs + 1e-8) - tf.math.log(-tf.math.log(u)), axis=-1, output_type=tf.int32)

        indices = tf.stack([tf.range(n_envs), actions], axis=1)
        log_probs = tf.math.log(tf.gather_nd(probs, indices) + 1e-8)

        force_sell = (positions == 1) & ((pnls <= -STOP_LOSS_PCT) | (pnls >= TAKE_PROFIT_PCT))
        effective_actions = tf.where(force_sell, 2, actions)

        next_close = closes[i + 1]
        price_ret  = (next_close - close) / close
        pred_ret   = forecast[0]                 # probability in [0, 1]
        pred_strength = pred_ret - 0.5           # signed confidence

        rewards = tf.zeros([n_envs], dtype=tf.float32)

        is_flat   = (positions == 0)
        flat_buy  = is_flat & (effective_actions == 1)
        flat_hold = is_flat & (effective_actions != 1)

        is_in     = (positions == 1)
        in_sell   = is_in & (effective_actions == 2)
        in_hold   = is_in & (effective_actions != 2)

        base_buy = price_ret - current_fee_rate - SLIPPAGE_PCT
        base_buy = tf.where(pred_strength > 0.0, base_buy + pred_strength * OPPORTUNITY_COST, base_buy)
        base_hold = tf.where(pred_strength > PRED_STRENGTH_OPPORTUNITY_TH, -pred_strength * OPPORTUNITY_COST * 0.3, 0.0)

        realized = pnls
        base_sell = realized - 2.0 * current_fee_rate - 2.0 * SLIPPAGE_PCT
        base_sell = tf.where(realized > 0.0, base_sell + realized * PROFIT_BONUS, base_sell)
        base_sell = tf.where((realized < -LOSS_CUT_THRESHOLD) & (realized > -STOP_LOSS_PCT), base_sell + LOSS_CUT_BONUS, base_sell)

        time_factor = tf.minimum(tf.cast(bars, tf.float32) / 20.0, 1.0)
        hold_penalty = -tf.abs(pnls) * HOLD_LOSER_DECAY * time_factor
        base_in_hold = tf.where(pnls > 0.0, price_ret * HOLD_WINNER_BONUS, price_ret + hold_penalty)

        rewards = tf.where(flat_buy, base_buy, rewards)
        rewards = tf.where(flat_hold, base_hold, rewards)
        rewards = tf.where(in_sell, base_sell, rewards)
        rewards = tf.where(in_hold, base_in_hold, rewards)

        positions = tf.where(flat_buy, 1, positions)
        positions = tf.where(in_sell,  0, positions)

        entry_prices = tf.where(flat_buy, close_batch, entry_prices)
        entry_prices = tf.where(in_sell,  0.0,         entry_prices)

        bars = tf.where(flat_buy, 0, bars)
        bars = tf.where(in_sell,  0, bars)
        bars = tf.where(in_hold,  bars + 1, bars)

        states_ta    = states_ta.write(idx, states)
        actions_ta   = actions_ta.write(idx, actions)
        log_probs_ta = log_probs_ta.write(idx, log_probs)
        values_ta    = values_ta.write(idx, values)
        rewards_ta   = rewards_ta.write(idx, rewards)

    return states_ta.stack(), actions_ta.stack(), log_probs_ta.stack(), values_ta.stack(), rewards_ta.stack()

print("Vectorized episode runner V2 OK (classification inputs & dynamic fees).")

In [ ]:
# PPO V2 training loop with Distributed Multi-GPU Strategy + best checkpoint + Fee Warmup

import os
os.makedirs(f"{WORK_DIR}/models/ppo/policy", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models/ppo/value",  exist_ok=True)

TOTAL_UPDATES = 1000
EARLY_STOP_PATIENCE_PPO = 10      # stop after this many post-warmup updates without a real improvement
MIN_DELTA_FRAC = 0.0025           # an update only counts as improvement if avg_reward beats
                                  # the best by >0.25% (relative). Filters out noise plateaus.
FEE_WARMUP_UPDATES = 20           # Curriculum learning: linearly increase fees over 20 updates

print(f"Training PPO V2 -- {TOTAL_UPDATES} updates x {NUM_PARALLEL} parallel episodes...")
print(f"Reward: no idle penalty | profit_bonus={PROFIT_BONUS} | entropy={ENTROPY_COEFF}")
print(f"Curriculum: Fee Warmup applied over {FEE_WARMUP_UPDATES} updates.")
print(f"Batched Mode: Distributed Multi-GPU Strategy | Batch: {PPO_BATCH_SIZE}")
print(f"Early stop after {EARLY_STOP_PATIENCE_PPO} post-warmup updates without >{MIN_DELTA_FRAC:.2%} improvement.")
print()

best_avg_reward = -float('inf')
updates_since_best = 0

for update in range(TOTAL_UPDATES):
    t0 = time.time()

    # Calculate current fee (Curriculum Learning)
    if update < FEE_WARMUP_UPDATES:
        current_fee = FEE_RATE * (update / FEE_WARMUP_UPDATES)
    else:
        current_fee = FEE_RATE

    # 1. Generate experiences for all 512 environments using the simulator
    st, ac, lp, va, rw = tf_run_episodes_vectorized(
        tf.constant(agent_candles[:, 4], dtype=tf.float32),
        tf.constant(agent_candles[:, 1], dtype=tf.float32),
        tf.constant(agent_candles[:, 2], dtype=tf.float32),
        tf.constant(agent_candles[:, 3], dtype=tf.float32),
        tf.constant(agent_forecasts, dtype=tf.float32),
        tf.constant(agent_vol_samples, dtype=tf.float32),
        tf.constant(NUM_PARALLEL, dtype=tf.int32),
        tf.constant(len(agent_forecasts), dtype=tf.int32),
        tf.constant(current_fee, dtype=tf.float32)  # Pass dynamic fee here
    )

    st_np = st.numpy()
    ac_np = ac.numpy()
    lp_np = lp.numpy()
    va_np = va.numpy()
    rw_np = rw.numpy()

    terminals = np.zeros((len(agent_forecasts), NUM_PARALLEL), dtype=bool)
    terminals[-1, :] = True

    # 2. Batched GPU Update
    pl, pe, vl = ppo_update_batched(st_np, ac_np, lp_np, va_np, rw_np, terminals)

    elapsed = time.time() - t0
    avg_reward = np.mean(np.sum(rw_np, axis=0))
    avg_buys   = np.mean(np.sum(ac_np == 1, axis=0))
    avg_sells  = np.mean(np.sum(ac_np == 2, axis=0))

    # ── Improvement tracking ──────────────────────────────────────────────
    # During fee warmup the reward is on a moving target (fees ramping up),
    # so we do NOT track best/early-stop until the warmup completes. Without
    # this guard, update 1 (fee=0) is always the "best" and every subsequent
    # (higher-fee) update looks worse -> early-stop fires before the real
    # training begins.
    in_warmup = update < FEE_WARMUP_UPDATES
    marker = ""
    if in_warmup:
        marker = " (warmup)"
    elif update == FEE_WARMUP_UPDATES:
        # First post-warmup update: reset tracking so the warmup-phase rewards
        # do not poison the comparison; this becomes the new baseline.
        best_avg_reward = avg_reward
        updates_since_best = 0
        marker = " * BEST (post-warmup baseline)"
        actor.save(f"{WORK_DIR}/models/ppo/policy/best_model.keras")
        critic.save(f"{WORK_DIR}/models/ppo/value/best_model.keras")
    else:
        threshold = best_avg_reward + max(abs(best_avg_reward), 1.0) * MIN_DELTA_FRAC
        if avg_reward > threshold:
            best_avg_reward = avg_reward
            updates_since_best = 0
            marker = " * BEST"
            actor.save(f"{WORK_DIR}/models/ppo/policy/best_model.keras")
            critic.save(f"{WORK_DIR}/models/ppo/value/best_model.keras")
        else:
            updates_since_best += 1

    plateau_warn = ""
    if (not in_warmup) and updates_since_best >= EARLY_STOP_PATIENCE_PPO:
        plateau_warn = f" [PLATEAU {updates_since_best}/{EARLY_STOP_PATIENCE_PPO}]"

    fee_info = f" | fee={current_fee:.5f}" if update <= FEE_WARMUP_UPDATES else ""

    print(f"  Update {update+1:02d}/{TOTAL_UPDATES}{fee_info} | avg_reward={avg_reward:+.4f} | "
          f"buys={avg_buys:.1f} sells={avg_sells:.1f} | {elapsed:.0f}s{marker}{plateau_warn}")

    # 3. Early stop: plateau reached AFTER warmup, no real improvement for PATIENCE updates
    if (not in_warmup) and updates_since_best >= EARLY_STOP_PATIENCE_PPO:
        print()
        print(f"  Early stopping at update {update+1}: no >{MIN_DELTA_FRAC:.2%} "
              f"improvement in {EARLY_STOP_PATIENCE_PPO} updates. Best checkpoint kept.")
        break

print()
print(f"PPO V2 trained. Best avg_reward: {best_avg_reward:+.4f}")

In [ ]:
# ── Save final models (PPO uses best checkpoint) ─────────────────────────────

import os, shutil
os.makedirs(f"{WORK_DIR}/models/bilstm", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models/ppo/policy", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models/ppo/value",  exist_ok=True)

# BiLSTM: save current state (already has best weights via restore_best_weights)
forecaster.save(f"{WORK_DIR}/models/bilstm/model.keras")
if 'full_forecaster' in globals():
    full_forecaster.save(f"{WORK_DIR}/models/bilstm/full_model.keras")

# PPO: use the best checkpoint saved during training
shutil.copy(f"{WORK_DIR}/models/ppo/policy/best_model.keras",
            f"{WORK_DIR}/models/ppo/policy/model.keras")
shutil.copy(f"{WORK_DIR}/models/ppo/value/best_model.keras",
            f"{WORK_DIR}/models/ppo/value/model.keras")

print(f"Models saved in {WORK_DIR}/models/")
print("  bilstm/model.keras (Baseline)")
print("  bilstm/full_model.keras (CNN-BiLSTM-MHA)")
print("  ppo/policy/model.keras  (best checkpoint)")
print("  ppo/value/model.keras   (best checkpoint)")


## Export to TensorFlow.js

All three models are exported as **`tfjs_graph_model`** (frozen computation graph).
This avoids the Keras 3 layers-model format that `tfjs-layers` (v4.x) cannot deserialize
(error: `Corrupted configuration, expected array for nodeData`).

Graph models are inference-only, which is fine here — training happens on Kaggle, the
bot only runs `predict()` in Node.

The final `tfjs_models.zip` contains everything needed to run the bot:
```
tfjs_models/
  bilstm/model.json        <- graph model (MultiHeadAttentionBlock baked in)
  ppo/
    policy/model.json      <- actor (graph model)
    value/model.json       <- critic (graph model)
```
**Setup:** download `tfjs_models.zip` from the Kaggle output panel and extract into `./models/`.


In [ ]:
# ── Export to TensorFlow.js (all models as graph_model) ──────────────────────
import os
import shutil, subprocess
import json
import base64

OUTPUT_BASE = f"{WORK_DIR}/tfjs_models"
os.makedirs(OUTPUT_BASE, exist_ok=True)

def _unfuse_tanh(model_json_path):
    """tfjs-node can't run fused _FusedMatMul[Tanh]; split into MatMul -> BiasAdd -> Tanh.
    The Tanh keeps the original node name so downstream inputs still resolve."""
    b64_tanh = base64.b64encode(b'Tanh').decode()
    b64_nhwc = base64.b64encode(b'NHWC').decode()
    with open(model_json_path) as f:
        m = json.load(f)
    out, patched = [], 0
    for n in m['modelTopology']['node']:
        fused = n.get('attr', {}).get('fused_ops', {}).get('list', {}).get('s', [])
        if n.get('op') != '_FusedMatMul' or b64_tanh not in fused:
            out.append(n); continue
        a, b, bias = n['input'][0], n['input'][1], n['input'][2]
        dev, base = n.get('device', ''), n['name']
        t  = n['attr'].get('T', {'type': 'DT_FLOAT'})
        ta = n['attr'].get('transpose_a', {'b': False})
        tb = n['attr'].get('transpose_b', {'b': False})
        mm, ba = base + '/MatMul_unfused', base + '/BiasAdd_unfused'
        out.append({'name': mm, 'op': 'MatMul', 'input': [a, b], 'device': dev,
                    'attr': {'transpose_a': ta, 'transpose_b': tb, 'T': t}})
        out.append({'name': ba, 'op': 'BiasAdd', 'input': [mm, bias], 'device': dev,
                    'attr': {'T': t, 'data_format': {'s': b64_nhwc}}})
        out.append({'name': base, 'op': 'Tanh', 'input': [ba], 'device': dev, 'attr': {'T': t}})
        patched += 1
    m['modelTopology']['node'] = out
    with open(model_json_path, 'w') as f:
        json.dump(m, f)
    print(f'  un-fused {patched} Tanh node(s) in {model_json_path}')

# ── BiLSTM: rebuild on CPU and export as SavedModel ─────────────────────────
# The new architecture uses only native Keras layers (Conv1D, BiLSTM,
# MultiHeadAttention, LayerNorm) so NO custom_objects are needed.
# We still need to re-build on CPU to avoid CudnnRNN ops in the SavedModel.

cpu_keras_path = f"{WORK_DIR}/models/bilstm/cpu_model.keras"
cpu_saved_path = f"{WORK_DIR}/models/bilstm/cpu_saved_model"

# Rebuild the exact same architecture on CPU, copy weights
cpu_forecaster = build_bilstm(WINDOW_SIZE, NUM_FEATURES, HIDDEN_UNITS, DROPOUT, HORIZON, L2)
cpu_forecaster.set_weights(forecaster.get_weights())
cpu_forecaster.save(cpu_keras_path)
print("BiLSTM saved as .keras (CPU copy)")

# Inline script runs conversion without GPU to avoid CudnnRNN in the SavedModel.
# NOTE: MultiHeadAttentionBlock is a custom layer, so we must register it.
convert_script = f"""
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import math, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

class MultiHeadAttentionBlock(layers.Layer):
    def __init__(self, d_model, num_heads=4, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model   = d_model
        self.num_heads = num_heads
        self.drop_rate = dropout
    def build(self, input_shape):
        feat_dim = input_shape[-1]
        self.needs_projection = (feat_dim != self.d_model)
        if self.needs_projection:
            self.input_proj = layers.Dense(self.d_model, name='mha_input_proj')
        self.mha = layers.MultiHeadAttention(
            num_heads=self.num_heads, key_dim=self.d_model // self.num_heads,
            dropout=self.drop_rate, name='mha_core')
        self.layernorm = layers.LayerNormalization(name='mha_layernorm')
        self.dropout   = layers.Dropout(self.drop_rate)
        super().build(input_shape)
    def call(self, x, training=None):
        residual = self.input_proj(x) if self.needs_projection else x
        attn_out = self.mha(query=x, value=x, key=x, training=training)
        attn_out = self.dropout(attn_out, training=training)
        return self.layernorm(residual + attn_out)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({{'d_model': self.d_model, 'num_heads': self.num_heads, 'dropout': self.drop_rate}})
        return cfg

model = keras.models.load_model('{cpu_keras_path}', custom_objects={{'MultiHeadAttentionBlock': MultiHeadAttentionBlock}}, compile=False)
model.export('{cpu_saved_path}')
print('SavedModel exported on CPU (no CudnnRNN)')
"""

with open('/tmp/convert_bilstm.py', 'w') as f:
    f.write(convert_script)

result = subprocess.run(
    ['python', '/tmp/convert_bilstm.py'],
    capture_output=True, text=True,
    env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''}
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

# Convert BiLSTM SavedModel -> tfjs_graph_model
result2 = subprocess.run([
    'tensorflowjs_converter',
    '--input_format=tf_saved_model',
    '--output_format=tfjs_graph_model',
    '--signature_name=serving_default',
    cpu_saved_path,
    f'{OUTPUT_BASE}/bilstm'
], capture_output=True, text=True,
   env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''})
print(result2.stdout)
if result2.returncode != 0:
    print('STDERR:', result2.stderr[-500:])
else:
    print('BiLSTM converted to tfjs_graph_model')
    # --- PATCH: Remove control dependencies from TF.js model.json ---
    model_json_path = f'{OUTPUT_BASE}/bilstm/model.json'
    with open(model_json_path, 'r') as f:
        m_data = json.load(f)
    for node in m_data.get('modelTopology', {}).get('node', []):
        if 'input' in node:
            node['input'] = [i for i in node['input'] if not i.startswith('^')]
    with open(model_json_path, 'w') as f:
        json.dump(m_data, f)
    print('Control dependencies stripped from BiLSTM model.json')
    _unfuse_tanh(model_json_path)

# ── Actor / Critic: SavedModel -> tfjs_graph_model ───────────────────────────

def export_keras_to_tfjs_graph(model_obj, name):
    saved_path = f"{WORK_DIR}/models/ppo/{name}/saved_model"
    if os.path.exists(saved_path):
        shutil.rmtree(saved_path)
    model_obj.export(saved_path)

    out_path = f"{OUTPUT_BASE}/ppo/{name}"
    res = subprocess.run([
        'tensorflowjs_converter',
        '--input_format=tf_saved_model',
        '--output_format=tfjs_graph_model',
        '--signature_name=serving_default',
        saved_path,
        out_path
    ], capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print(f'STDERR ({name}):', res.stderr[-500:])
        return False
    print(f'  {name} -> tfjs_graph_model OK')
    _unfuse_tanh(f'{out_path}/model.json')
    return True

export_keras_to_tfjs_graph(actor,  'policy')
export_keras_to_tfjs_graph(critic, 'value')

# Sanity check
for sub in ['bilstm', 'ppo/policy', 'ppo/value']:
    with open(f'{OUTPUT_BASE}/{sub}/model.json', 'r') as f:
        fmt = json.load(f).get('format', '<missing>')
    print(f'  {sub}: format = {fmt}')
    assert fmt == 'graph-model', f'{sub} expected graph-model, got {fmt}'

shutil.make_archive(f'{WORK_DIR}/tfjs_models', 'zip', OUTPUT_BASE)
print(f'\nFinal file: {WORK_DIR}/tfjs_models.zip')
subprocess.run(['find', OUTPUT_BASE, '-type', 'f'], check=True)


In [ ]:
# ── Validation Visualization ─────────────────────────────────────────
# Selects 3 random scenarios from the validation set to
# compare the model's prediction (red line) with what actually
# happened (green line, actual soft labels).

import matplotlib.pyplot as plt
import numpy as np

print("Generating validation charts...")
# Choose 3 different scenarios from the Validation Set
np.random.seed(42)
indices = np.random.choice(len(X_val), 3, replace=False)

y_pred = forecaster(X_val[indices], training=False).numpy()

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Model Validation: 3 Scenarios (Prediction vs. Reality)', fontsize=18)

for i, idx in enumerate(indices):
    ax = axes[i]
    # Price history (Normalized) for a 128-period window.
    price_history = X_val[idx, :, 0]

    # History plot
    ax.plot(range(len(price_history)), price_history, label='Historical Price (Normal)', color='blue', linewidth=1.5)

    # X-axis for the future (Predictive horizon)
    start_future = len(price_history)
    future_x = range(start_future, start_future + HORIZON)

    actual_labels = y_val[idx]
    pred_labels = y_pred[i]

    # Second Y-axis for probabilities (Soft Labels at [0, 1])
    ax2 = ax.twinx()
    ax2.plot(future_x, actual_labels, marker='o', markersize=6, label='Reality (Soft Label)', color='green', linestyle='--')
    ax2.plot(future_x, pred_labels, marker='X', markersize=8, label='Model Prediction', color='red', linestyle='-')

    # Formatting and Captions
    ax.set_title(f'Scenario {i+1} (Validation Sample #{idx})', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel('Standardized Price')
    ax2.set_ylabel('P(High) - Soft Label')
    ax2.set_ylim(-0.05, 1.05)

    # Neutral line of 0.5 (no strong trend)
    ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.6)

    # Unifying captions from both axes
    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper left')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

## Loading in TypeScript

Kaggle notebook: [https://www.kaggle.com/code/acaurangel/bilstm-ppo-self-attention-ai-spot-trading](https://www.kaggle.com/code/acaurangel/bilstm-ppo-self-attention-ai-spot-trading)

**Setup:**
1. Download `tfjs_models.zip` from the Kaggle output panel after training.
2. Extract into `./models/` at the project root.

**Model formats:** all three are `tfjs_graph_model`, loaded with `tf.loadGraphModel`.

```typescript
const forecaster = await tf.loadGraphModel('file://./models/bilstm/model.json');
const actor      = await tf.loadGraphModel('file://./models/ppo/policy/model.json');
const critic     = await tf.loadGraphModel('file://./models/ppo/value/model.json');
```

## Testando a Inferência na Prática (20 Casos)

Aqui demonstramos o fluxo exato que o seu bot (em Python ou Node.js) fará em tempo real:
1. Coleta os últimos 128 candles e calcula as 16 features.
2. Passa as features pelo **Forecaster** para prever a direção (P(up) para 4 tempos).
3. Junta as previsões com o preço atual, volatilidade e estado da posição para formar o **State Vector** (13 features).
4. Passa o estado pelo **Actor** (para obter as probabilidades de ação) e pelo **Critic** (para o valor esperado).

In [ ]:
import numpy as np
import tensorflow as tf

print("Demonstrando a utilização dos modelos em 20 casos aleatórios:")
print("-" * 70)

# Pegar 20 índices aleatórios do conjunto do agente
np.random.seed(42)
sample_indices = np.random.choice(len(agent_X), 20, replace=False)

action_labels = ["HOLD (0)", "BUY (1)", "SELL (2)"]

for i, idx in enumerate(sample_indices):
    print(f"\nCaso {i+1} (Índice de tempo {idx}):")

    # 1. Obter a janela de features (128 candles, 16 features) e prever com o Forecaster
    seq_features = agent_X[idx : idx+1]
    forecast = forecaster(seq_features, training=False).numpy()[0]

    # Formatação amigável da previsão
    fc_str = ", ".join([f"{p:.2%}" for p in forecast])
    print(f"  [Forecaster] Previsão P(up) próximos 4 candles: [{fc_str}]")

    # --> NOVA PARTE: Calcular e mostrar a realidade (retorno futuro dos próximos 4 candles)
    candle_idx = idx + WINDOW_SIZE
    anchor_close = agent_candles[candle_idx, 4]
    real_returns = []
    for h in range(1, HORIZON + 1):
        if candle_idx + h < len(agent_candles):
            future_close = agent_candles[candle_idx + h, 4]
            ret = (future_close - anchor_close) / anchor_close
            real_returns.append(f"{ret:+.2%}")
        else:
            real_returns.append("N/A")

    real_str = ", ".join(real_returns)
    print(f"  [Realidade]  Retorno real nos próximos 4 candles: [{real_str}]")

    # 2. Montar o estado atual para o Agente PPO
    # Simularemos que o bot está FLAT (sem posição aberta)
    position = 0
    entry_price = 0.0
    bars_in_pos = 0

    volatility = agent_vol_samples[idx]

    state_vec = assemble_state(
        agent_candles,
        candle_idx,
        forecast,
        position,
        entry_price,
        bars_in_pos,
        volatility
    )

    state_tensor = tf.convert_to_tensor([state_vec], dtype=tf.float32)

    # 3. Consultar o Ator (Policy) e o Crítico (Value)
    action_probs = actor(state_tensor, training=False).numpy()[0]
    value = critic(state_tensor, training=False).numpy()[0, 0]

    chosen_action = np.argmax(action_probs)

    print(f"  [Critic]     Valor estimado do estado (V-value): {value:+.4f}")
    print(f"  [Actor]      Probabilidades: HOLD={action_probs[0]:.2%}, BUY={action_probs[1]:.2%}, SELL={action_probs[2]:.2%}")
    print(f"  => Ação Recomendada: {action_labels[chosen_action]}")

In [ ]:
import matplotlib.pyplot as plt

print("Gerando gráficos de avaliação para os 20 casos...")

fig, axes = plt.subplots(20, 1, figsize=(12, 80))
fig.suptitle('Avaliação de Inferência: 20 Casos Aleatórios', fontsize=18, fontweight='bold')

# Vamos dar um zoom nos últimos 40 candles do passado para visualizar melhor
ZOOM_PAST = 40

for i, idx in enumerate(sample_indices):
    ax = axes[i]
    candle_idx = idx + WINDOW_SIZE

    # Preços para plotagem
    past_closes = agent_candles[idx : candle_idx + 1, 4]  # Inclui o candle atual (tempo 0)
    future_closes = agent_candles[candle_idx : candle_idx + HORIZON + 1, 4]

    # Obter previsão e ação recomendada novamente para o plot
    seq_features = agent_X[idx : idx+1]
    forecast = forecaster(seq_features, training=False).numpy()[0]

    state_vec = assemble_state(
        agent_candles, candle_idx, forecast, 0, 0.0, 0, agent_vol_samples[idx]
    )
    state_tensor = tf.convert_to_tensor([state_vec], dtype=tf.float32)
    action_probs = actor(state_tensor, training=False).numpy()[0]
    chosen_action = np.argmax(action_probs)
    action_name = action_labels[chosen_action]

    # Lógica de Avaliação (Correto/Errado)
    # Retorno máximo nos próximos 4 candles
    future_returns = (future_closes[1:] - future_closes[0]) / future_closes[0]
    max_future_ret = np.max(future_returns)

    # Vamos considerar que para lucrar (BUY) precisamos de pelo menos +0.1% para bater as taxas.
    # Se ficarmos de fora (SELL ou HOLD), o modelo acerta se não houver um grande movimento de alta.
    PROFIT_THRESHOLD = 0.001

    if chosen_action in [0, 2]: # HOLD ou SELL (Estratégia defensiva / Ficar de fora)
        if max_future_ret <= PROFIT_THRESHOLD:
            eval_label = "CORRECT (Evitou perda / Sem direção clara)"
            color = "green"
        else:
            eval_label = "WRONG (Perdeu oportunidade de lucro)"
            color = "red"
    else: # BUY (Comprar)
        if max_future_ret > PROFIT_THRESHOLD:
            eval_label = "CORRECT (Capturou lucro)"
            color = "green"
        else:
            eval_label = "WRONG (Comprou e caiu/ficou flat)"
            color = "red"

    # Eixos X
    x_past = np.arange(-WINDOW_SIZE, 1)
    x_future = np.arange(0, HORIZON + 1)

    # Plot
    ax.plot(x_past[-ZOOM_PAST-1:], past_closes[-ZOOM_PAST-1:], label='Passado (Zoom)', color='blue', linewidth=2)
    ax.plot(x_future, future_closes, label='Futuro Real', color='orange', linestyle='--', linewidth=2, marker='o')
    ax.axvline(0, color='black', linestyle=':', alpha=0.5, label='Momento da Decisão')

    # Formatação
    ax.set_title(f"Caso {i+1} | Ação: {action_name} | Status: {eval_label}", color=color, fontweight='bold', fontsize=12)
    ax.set_ylabel("Preço (USDT)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left")

plt.tight_layout()
plt.subplots_adjust(top=0.98)
plt.show()